# Topic 8 — Conversation Memory: The SupportAgentMemory Class

Capstone: everything from topics 1-7, wrapped into one clean class a real support agent could actually call.

In [ ]:
import redis
from google.cloud.firestore_v1 import ArrayUnion
from setup import firestore_client

class SupportAgentMemory:
    def __init__(self, session_id: str, customer_id: str, redis_host="localhost", redis_port=6379):
        self.session_id = session_id
        self.customer_id = customer_id
        self.redis = redis.Redis(host=redis_host, port=redis_port, decode_responses=True)
        self.profile_doc = firestore_client.collection("support_customer_profiles").document(customer_id)
        self.session_key = f"support_session:{session_id}:turns"

    def remember_turn(self, role: str, content: str):
        self.redis.rpush(self.session_key, f"{role}: {content}")
        self.redis.expire(self.session_key, 1800)

    def remember_fact(self, fact: str):
        self.profile_doc.set({"known_issues": ArrayUnion([fact])}, merge=True)

    def recall(self) -> str:
        turns = self.redis.lrange(self.session_key, 0, -1)
        profile = self.profile_doc.get().to_dict() or {}
        return (
            "Current chat:\n" + "\n".join(turns) +
            f"\n\nKnown issues: {profile.get('known_issues', [])}"
        )

### Use it the way a real agent would

In [ ]:
memory = SupportAgentMemory(session_id="sess_9001", customer_id="cust_042")

memory.remember_turn("customer", "My app keeps crashing when I open settings.")
memory.remember_fact("App crashes on opening settings - reported today")

print(memory.recall())

### This is what would get passed into SupportBot's next Gemini call

In [ ]:
context = memory.recall()
print("\n--- This context feeds directly into the model's prompt ---\n")
print(context)